In [0]:
%sql

DESCRIBE bakery_pos_develop.silver.product_product

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_product AS SELECT
    product_id,
    name,
    categ_id,
    categ_name,
    lst_price,
    taxes_id,
    default_code,
    standard_price,
    created_at,
    updated_at,
    is_updated,
    is_deleted
FROM bakery_pos_develop.silver.product_product
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY product_id
    ORDER BY updated_at DESC
) = 1

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.dim_product_product (
    product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id BIGINT,
    name STRING,
    categ_id INT,
    categ_name STRING,
    lst_price DOUBLE,
    taxes_id ARRAY<STRING>,
    default_code STRING,
    standard_price DOUBLE,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN,
    PRIMARY KEY (product_sk)
)

In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.dim_product_product AS tgt
USING stg_product AS src
ON tgt.product_id = src.product_id

WHEN MATCHED AND src.updated_at > tgt.updated_at THEN
    UPDATE SET
        tgt.name = src.name,
        tgt.categ_id = src.categ_id,
        tgt.categ_name = src.categ_name,
        tgt.lst_price = src.lst_price,
        tgt.taxes_id = src.taxes_id,
        tgt.default_code = src.default_code,
        tgt.standard_price = src.standard_price,
        tgt.created_at = src.created_at,
        tgt.updated_at = src.updated_at,
        tgt.is_updated = src.is_updated,
        tgt.is_deleted = src.is_deleted

WHEN NOT MATCHED THEN
    INSERT (
        product_id,
        name,
        categ_id,
        categ_name,
        lst_price,
        taxes_id,
        default_code,
        standard_price,
        created_at,
        updated_at,
        is_updated,
        is_deleted
    )
    VALUES (
        src.product_id,
        src.name,
        src.categ_id,
        src.categ_name,
        src.lst_price,
        src.taxes_id,
        src.default_code,
        src.standard_price,
        src.created_at,
        src.updated_at,
        src.is_updated,
        src.is_deleted
    )